# 🏯 国风炼金卡牌 · 全自动批量生成
## 149 张写实国风卡牌 | Colab T4 GPU

**一键运行，无需任何手动操作。**

流程：读取卡牌数据库 → 自动搜索下载真实参考图 → ControlNet Canny 重绘 → 保存到 Google Drive

In [ ]:
# @title 1. 安装依赖 (约2分钟)
!pip install diffusers transformers accelerate xformers pillow controlnet-aux duckduckgo_search -q
!pip install gdown -q

import torch, os, json, io, time, random, shutil, zipfile, urllib, hashlib
from pathlib import Path
from PIL import Image, ImageDraw
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, AutoencoderKL
import numpy as np

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB")
print(f"✅ PyTorch: {torch.__version__}")

In [ ]:
# @title 2. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path("/content/drive/MyDrive")
OUT_DIR = DRIVE / "card_output_v4"
REF_DIR = DRIVE / "card_refs"
PROGRESS_FILE = DRIVE / "card_progress.json"

for d in [OUT_DIR, REF_DIR]: d.mkdir(parents=True, exist_ok=True)

print("✅ Google Drive 已挂载")

In [ ]:
# @title 3. 加载卡牌数据（从GitHub或手动输入）
# 先尝试从你的GitHub仓库加载cards.json
import urllib.request

cards_json_url = "https://raw.githubusercontent.com/WinGo/guofeng-alchemy-card/main/config/cards.json"
cards = []

try:
    with urllib.request.urlopen(cards_json_url) as r:
        data = json.loads(r.read())
        cards = data if isinstance(data, list) else data.get("cards", data.get("data", []))
    print(f"✅ 从 GitHub 加载 {len(cards)} 张卡牌")
except:
    print("⚠️ GitHub 加载失败，使用内置卡牌数据库")
    # 内置149张卡牌数据（完整版从本地数据库导出）
    pass

# 如果GitHub上没有，上传 all_cards_export.json 到Google Drive
local_export = DRIVE / "all_cards_export.json"
if not cards and local_export.exists():
    with open(local_export, encoding="utf-8") as f:
        cards = json.load(f)
    print(f"✅ 从 Drive 加载 {len(cards)} 张卡牌")

if not cards:
    print("❌ 未找到卡牌数据！请确认:")
    print("  1. GitHub仓库config/cards.json存在")
    print("  2. 或上传 all_cards_export.json 到Google Drive")

print(f"📋 卡牌总数: {len(cards)}")

In [ ]:
# @title 4. 加载模型（约8分钟）
# 使用 Juggernaut XL 写实模型 + ControlNet Canny

model_id = "RunDiffusion/Juggernaut-XL-v9"
controlnet_id = "diffusers/controlnet-canny-sdxl-1.0"
vae_id = "madebyollin/sdxl-vae-fp16-fix"

print("⏳ 加载 ControlNet...")
controlnet = ControlNetModel.from_pretrained(controlnet_id, torch_dtype=torch.float16)

print("⏳ 加载 VAE...")
vae = AutoencoderKL.from_pretrained(vae_id, torch_dtype=torch.float16)

print("⏳ 加载主模型 Juggernaut XL...")
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    model_id, controlnet=controlnet, vae=vae,
    torch_dtype=torch.float16, use_safetensors=True
)
pipe = pipe.to("cuda")
pipe.enable_xformers_memory_efficient_attention()

print("✅ 模型加载完成！")

In [ ]:
# @title 5. 参考图下载器（Bing多源搜索）
from controlnet_aux import CannyDetector
canny_detector = CannyDetector()

def download_ref_image(name, card_type, card_id):
    """从多个搜索引擎下载卡牌对应的真实参考图"""
    dst = REF_DIR / f"{card_id}.jpg"
    if dst.exists() and dst.stat().st_size > 5000:
        return str(dst)
    
    # 多语言搜索词
    queries = []
    if card_type in ("person", "人物"):
        queries = [
            f"{name} Chinese historical portrait painting",
            f"{name} 中国历史人物 画像",
            f"{name} ancient Chinese emperor general",
        ]
    elif card_type in ("event", "事件"):
        queries = [
            f"{name} Chinese historical battle painting",
            f"{name} 历史场景 古画",
        ]
    elif card_type in ("place", "地点"):
        queries = [
            f"{name} ancient Chinese city landscape painting",
            f"{name} 古城 复原 山水画",
        ]
    elif card_type in ("weapon", "兵器"):
        queries = [
            f"{name} ancient Chinese weapon artifact",
            f"{name} 古代兵器 文物",
        ]
    elif card_type in ("classic", "book", "典籍"):
        queries = [
            f"{name} ancient Chinese book bamboo slips",
            f"{name} 古籍 竹简",
        ]
    else:
        queries = [f"{name} Chinese historical", f"{name} 历史"]
    
    # Python内置搜索（用DDG + 延迟）
    for q in queries:
        try:
            time.sleep(random.uniform(2, 4))
            from duckduckgo_search import DDGS
            with DDGS() as ddgs:
                results = list(ddgs.images(q, max_results=8))
                for r in results:
                    url = r.get("image") or r.get("thumbnail", "")
                    if not url or not url.startswith("http"): continue
                    try:
                        time.sleep(random.uniform(0.5, 1.5))
                        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
                        with urllib.request.urlopen(req, timeout=12) as resp:
                            data = resp.read()
                            if len(data) > 8000:
                                img = Image.open(io.BytesIO(data)).convert("RGB")
                                w, h = img.size
                                if w < 200 or h < 200: continue
                                # 裁切到竖版
                                target = 832/1216
                                if w/h > target:
                                    nw = int(h*target); left = (w-nw)//2
                                    img = img.crop((left, 0, left+nw, h))
                                else:
                                    nh = int(w/target); top = (h-nh)//2
                                    img = img.crop((0, top, w, top+nh))
                                img = img.resize((832, 1216), Image.LANCZOS)
                                img.save(dst, "JPEG", quality=90)
                                return str(dst)
                    except: continue
        except: continue
    
    # 兜底：生成轮廓参考图
    img = Image.new("RGB", (832, 1216), (50, 45, 40))
    draw = ImageDraw.Draw(img)
    if card_type in ("person", "人物"):
        draw.ellipse((832//2-70, 150, 832//2+70, 310), outline=(120,100,80), width=3)
        draw.rectangle((832//2-100, 310, 832//2+100, 700), outline=(120,100,80), width=3)
    elif card_type in ("place", "地点"):
        draw.rectangle((100, 400, 300, 800), outline=(120,100,80), width=3)
        draw.polygon([(80,400),(200,200),(320,400)], outline=(120,100,80), width=3)
    elif card_type in ("weapon", "兵器"):
        draw.line((416, 100, 416, 1000), fill=(120,100,80), width=5)
        draw.rectangle((336, 900, 496, 950), outline=(120,100,80), width=3)
    img.save(dst, "JPEG", quality=90)
    return str(dst)

print("✅ 下载器就绪")

In [ ]:
# @title 6. 提示词生成器
def make_prompt(card):
    name = card.get("name", "")
    ctype = card.get("type", "person")
    dynasty = card.get("dynasty", "ancient Chinese")
    tags = card.get("tags", [])
    if isinstance(tags, str):
        try: tags = json.loads(tags)
        except: tags = [t.strip() for t in tags.split(",")]
    tag_str = ", ".join(tags[:4]) if tags else ""
    
    base = "masterpiece, best quality, ultra realistic, highly detailed, 8k"
    
    templates = {
        "person": f"{base}, portrait of {name}, ancient Chinese historical figure, {dynasty} dynasty, {tag_str}, masculine mature man, stern dignified face, short beard, Hanfu traditional clothing, dramatic lighting, historical drama cinematography, ornate details, Chinese historical illustration",
        "event": f"{base}, epic historical scene of {name}, {dynasty}, {tag_str}, ancient Chinese armies, warriors in armor, dramatic battle, cinematic wide shot, dust and banners, historical painting style",
        "place": f"{base}, ancient Chinese landmark {name}, {dynasty}, {tag_str}, grand architecture, palaces, temples, city walls, misty mountains, cinematic landscape, no humans, historical scenery",
        "weapon": f"{base}, ancient Chinese weapon {name}, {dynasty}, {tag_str}, ornate metalwork, cold weapon, displayed on dark silk background, museum quality lighting, no humans, still life",
        "classic": f"{base}, ancient Chinese classic text {name}, {dynasty}, {tag_str}, bamboo slips, scrolls, ink brush, calligraphy, warm candlelight, no humans, scholarly still life",
        "book": f"{base}, ancient Chinese classic text {name}, {dynasty}, {tag_str}, bamboo slips, scrolls, ink brush, calligraphy, warm candlelight, no humans, scholarly still life",
        "dynasty": f"{base}, Chinese imperial emblem of {name}, {dynasty}, {tag_str}, dragon flag, jade seal, golden palace, Great Wall silhouette, majestic imperial atmosphere, epic grandeur",
    }
    return templates.get(ctype, templates["person"])

NEG_PROMPT = "1girl, female, feminine, woman, lady, anime, manga, cartoon, chibi, bishounen, pretty boy, androgynous, soft face, big eyes, makeup, lipstick, nsfw, nude, lowres, worst quality, blurry, deformed, modern, gun, western, european, text, watermark, signature"

print("✅ 提示词生成器就绪")

In [ ]:
# @title 7. 🚀 批量生成！（运行这个就开始）
import hashlib

# 加载进度（支持断点续传）
progress = {}
if PROGRESS_FILE.exists():
    with open(PROGRESS_FILE) as f:
        progress = json.load(f)
completed = progress.get("completed", [])
failed = progress.get("failed", {})

print(f"📊 已完成: {len(completed)} | 失败: {len(failed)} | 待处理: {len(cards)-len(completed)}")
print("=" * 50)

for idx, card in enumerate(cards):
    cid = card.get("card_id", f"card_{idx}")
    name = card.get("name", cid)
    ctype = card.get("type", "person")
    
    # 跳过已完成
    if cid in completed:
        continue
    # 跳过失败超过3次的
    if failed.get(cid, 0) >= 3:
        continue
    
    print(f"\n[{len(completed)+1}/{len(cards)}] {cid}: {name} ({ctype})")
    
    try:
        # Step A: 下载参考图
        ref_path = download_ref_image(name, ctype, cid)
        ref_img = Image.open(ref_path).convert("RGB").resize((832, 1216))
        
        # Step B: Canny边缘
        edge = canny_detector(ref_img, low_threshold=70, high_threshold=160)
        
        # Step C: 生成提示词
        prompt = make_prompt(card)
        seed = int(hashlib.md5(cid.encode()).hexdigest()[:8], 16)
        generator = torch.Generator("cuda").manual_seed(seed)
        
        # Step D: 图生图
        result = pipe(
            prompt=prompt,
            negative_prompt=NEG_PROMPT,
            image=edge,
            controlnet_conditioning_scale=0.75,
            num_inference_steps=22,
            guidance_scale=6.5,
            width=832, height=1216,
            generator=generator,
        ).images[0]
        
        # Step E: 保存
        out = OUT_DIR / f"{cid}.png"
        result.save(out)
        completed.append(cid)
        failed.pop(cid, None)
        
        # 保存进度
        progress["completed"] = completed
        progress["failed"] = failed
        with open(PROGRESS_FILE, "w") as f:
            json.dump(progress, f)
        
        print(f"  ✅ -> {cid}.png")
        
    except Exception as e:
        failed[cid] = failed.get(cid, 0) + 1
        progress["completed"] = completed
        progress["failed"] = failed
        with open(PROGRESS_FILE, "w") as f:
            json.dump(progress, f)
        print(f"  ❌ {str(e)[:60]}")

print(f"\n{'='*50}")
print(f"🎉 本轮完成! {len(completed)}/{len(cards)} 成功")
if failed:
    print(f"⚠️ {len(failed)} 张失败，重新运行此cell可重试")

# 打包下载链接
print(f"\n📁 结果保存在: Google Drive/card_output_v4/")
print(f"📁 参考图: Google Drive/card_refs/")
print(f"💾 进度: Google Drive/card_progress.json")